# Notebook 08 — 10-Sekunden-Fenster (Live-App-Matching)

**Motivation:** Die Live-App klassifiziert auf 10-s-Fenstern (`WINDOW_SECS=10`, `SLIDE_SECS=2`).
NB04–07 trainierten auf 25-s-Fenstern → Mismatch in Feature-Statistiken.
Dieses Notebook trainiert auf **10-s-Fenstern** um Training- und Inferenz-Distribution anzugleichen.

| | NB04–07 | NB08 (diese) |
|--|--|--|
| Fensterlänge | 25 s | **10 s** |
| Step Training | 25 s (kein Overlap) | **5 s** (50 % Overlap) |
| Min-Länge | 20 s | **8 s** (= MIN_WINDOW in App) |
| Evaluation | LOO per Fenster | LOO per Fenster |

**Modelle:** RF · SVM · kNN · MLP (identisch NB05)

---
## 1. Setup

In [ ]:
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

CLASSES_RAW    = ['Apfel', 'Kaugummi', 'Skyr', 'Still', 'Essen']
CLASSES_COARSE = ['Still', 'Essen']
CLASSES_FINE   = ['Apfel', 'Kaugummi', 'Skyr', 'Essen']
CLASSES_FULL   = ['Still', 'Apfel', 'Kaugummi', 'Skyr', 'Essen']
TO_COARSE      = {'Apfel': 'Essen', 'Kaugummi': 'Essen',
                  'Skyr':  'Essen', 'Still':    'Still', 'Essen': 'Essen'}
COLORS = {'Apfel': '#e15759', 'Kaugummi': '#4e79a7', 'Skyr': '#f28e2b',
          'Still': '#59a14f', 'Essen': '#b07aa1'}

FS            = 50.0
TRIM_SECS     = 2
WINDOW_SECS   = 10.0   # ← live app
STEP_SECS     = 10.0   # kein Overlap
MIN_TAIL_SECS = 8.0    # ← MIN_WINDOW in classifier_app.py
K_ME          = 5
CV_FOLDS      = 10     # 10-Fold statt LOO — schnell und zuverlässig bei >500 Samples

# Pipelines: Scaler + Modell zusammen damit cross_val_predict korrekt skaliert
def make_pipeline(clf, scale=True):
    if scale:
        return Pipeline([('sc', StandardScaler()), ('clf', clf)])
    return clf

MODELS = {
    'RF':  dict(
        clf=lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
        scale=False),
    'SVM': dict(
        clf=lambda: SVC(kernel='rbf', C=10, class_weight='balanced', random_state=42),
        scale=True),
    'kNN': dict(
        clf=lambda: KNeighborsClassifier(n_neighbors=7, weights='distance', metric='euclidean'),
        scale=True),
    'MLP': dict(
        clf=lambda: MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                                   solver='adam', max_iter=500,
                                   learning_rate_init=0.001, random_state=42),
        scale=True),
}
MODEL_COLORS = {'RF': '#4e79a7', 'SVM': '#e15759', 'kNN': '#f28e2b', 'MLP': '#59a14f'}

---
## 2. Daten laden, segmentieren, Features extrahieren

In [ ]:
DATA_DIR = Path('../data/raw')
_SKIP    = {'Metadata.csv', 'Annotation.csv'}

def preprocess(df):
    df = df.copy()
    t  = df['seconds_elapsed']
    df = df[(t >= t.iloc[0] + TRIM_SECS) &
            (t <= t.iloc[-1] - TRIM_SECS)].reset_index(drop=True)
    df['lin_x']     = df['accelerationX']
    df['lin_y']     = df['accelerationY']
    df['lin_z']     = df['accelerationZ']
    df['magnitude'] = np.sqrt(df['lin_x']**2 + df['lin_y']**2 + df['lin_z']**2)
    return df

def sliding_windows(df):
    t = df['seconds_elapsed'].values
    t_start, t_end = t[0], t[-1]
    out = []
    while t_start + MIN_TAIL_SECS <= t_end:
        t_stop = t_start + WINDOW_SECS
        w = df[(t >= t_start) & (t < t_stop)].reset_index(drop=True)
        if len(w) > 1 and (w['seconds_elapsed'].iloc[-1] - w['seconds_elapsed'].iloc[0]) >= MIN_TAIL_SECS:
            out.append(w)
        t_start += STEP_SECS
    return out

def movement_mask(df):
    thr = max(0.02, K_ME * df['magnitude'].median())
    return df['magnitude'].rolling(50, center=True, min_periods=1).max() <= thr

def extract_features(df):
    feats = {}
    for col in ['lin_x', 'lin_y', 'lin_z', 'magnitude']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    feats['stillness_ratio'] = (df['magnitude'] < 0.02).mean()
    feats['movement_events'] = int((df['magnitude'] > df['magnitude'].quantile(0.75)).sum())
    for col in ['rotationRateX', 'rotationRateY', 'rotationRateZ']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    for col in ['pitch', 'roll', 'yaw']:
        feats[f'{col}_mean']  = df[col].mean()
        feats[f'{col}_std']   = df[col].std()
        feats[f'{col}_range'] = df[col].max() - df[col].min()
    nperseg = min(128, len(df) // 2)
    freqs, psd = welch(df['magnitude'].values, fs=FS, nperseg=nperseg)
    chew = (freqs >= 0.5) & (freqs <= 4.0)
    cf, cp = freqs[chew], psd[chew]
    feats['total_power']        = float(psd.sum())
    feats['chew_band_power']    = float(cp.sum())
    feats['rhythmicity']        = feats['chew_band_power'] / feats['total_power'] if feats['total_power'] > 0 else 0.0
    feats['dominant_chew_freq'] = float(cf[np.argmax(cp)]) if len(cp) > 0 else 0.0
    return feats

rows, y_raw = [], []
for zf in sorted(DATA_DIR.glob('*.zip')):
    for cls in CLASSES_RAW:
        if zf.name.startswith(cls + '_'):
            with zipfile.ZipFile(zf) as z:
                csv_name = next(f for f in z.namelist() if f.endswith('.csv') and f not in _SKIP)
                with z.open(csv_name) as f:
                    df = preprocess(pd.read_csv(f))
            for w in sliding_windows(df):
                clean = w[movement_mask(w)].reset_index(drop=True)
                if len(clean) > 30:
                    rows.append(extract_features(clean))
                    y_raw.append(cls)
            break

X_all    = pd.DataFrame(rows)
y_raw    = np.array(y_raw)
y_coarse = np.array([TO_COARSE[c] for c in y_raw])
eat_mask = y_coarse == 'Essen'
X_eat    = X_all[eat_mask].reset_index(drop=True)
y_eat    = y_raw[eat_mask]

print(f'Fenster ({WINDOW_SECS:.0f}s, Step={STEP_SECS:.0f}s, kein Deckel):')
print(f'  Gesamt: {len(X_all)}  (NB04/05 Referenz: 256)')
print(f'  Still:  {(~eat_mask).sum()}')
print(f'  Essen:  {eat_mask.sum()}  (NB04/05: 170)')
for cls in CLASSES_FINE:
    print(f'    {cls:10s}: {(y_raw==cls).sum()}')

---
## 3. LOO-CV — alle Modelle

LOO per Fenster (wie NB04/05) → vergleichbar mit Referenzwerten.

> **Hinweis:** 5-s-Step bedeutet 50 % Overlap zwischen benachbarten Fenstern.
> LOO per Fenster hat daher leichten Leakage-Effekt. Für ehrliche Evaluation
> → NB07 LOSO.

In [ ]:
def cv_eval(X_np, y_str, cfg):
    """10-Fold Stratified CV mit cross_val_predict. Gibt (yt, yp) zurück."""
    pipe = make_pipeline(cfg['clf'](), scale=cfg['scale'])
    cv   = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    yp   = cross_val_predict(pipe, X_np, y_str, cv=cv, n_jobs=-1)
    return y_str, yp

print(f'cv_eval() bereit — {CV_FOLDS}-Fold Stratified CV, n_jobs=-1')

In [ ]:
results_s1 = {}
for name, cfg in MODELS.items():
    print(f'S1 {name}…', end=' ', flush=True)
    yt, yp = cv_eval(X_all.values, y_coarse, cfg)
    results_s1[name] = {'yt': yt, 'yp': yp, 'acc': accuracy_score(yt, yp)}
    print(f'{results_s1[name]["acc"]:.1%}')

In [ ]:
results_s2 = {}
for name, cfg in MODELS.items():
    print(f'S2 {name}…', end=' ', flush=True)
    yt, yp = cv_eval(X_eat.values, y_eat, cfg)
    results_s2[name] = {'yt': yt, 'yp': yp, 'acc': accuracy_score(yt, yp)}
    print(f'{results_s2[name]["acc"]:.1%}')

In [ ]:
def e2e_cv(cfg):
    """End-to-End 10-Fold CV: S1 entscheidet, S2 klassifiziert Essen-Samples."""
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    yt_all, yp_all = [], []

    for tr, te in cv.split(X_all.values, y_coarse):
        # Stufe 1
        pipe1 = make_pipeline(cfg['clf'](), scale=cfg['scale'])
        pipe1.fit(X_all.values[tr], y_coarse[tr])
        preds_c = pipe1.predict(X_all.values[te])

        # Stufe 2 trainieren
        eat_tr = tr[y_coarse[tr] == 'Essen']
        if len(eat_tr) > 0 and len(np.unique(y_raw[eat_tr])) >= 2:
            pipe2 = make_pipeline(cfg['clf'](), scale=cfg['scale'])
            pipe2.fit(X_all.values[eat_tr], y_raw[eat_tr])
        else:
            pipe2 = None

        for i, pred_c in zip(te, preds_c):
            if pred_c == 'Essen' and pipe2 is not None:
                pred = pipe2.predict(X_all.values[[i]])[0]
            else:
                pred = pred_c
            yp_all.append(pred)
            yt_all.append(y_raw[i])

    return np.array(yt_all), np.array(yp_all)

results_e2e = {}
for name, cfg in MODELS.items():
    print(f'E2E {name}…', end=' ', flush=True)
    yt, yp = e2e_cv(cfg)
    results_e2e[name] = {'yt': yt, 'yp': yp, 'acc': accuracy_score(yt, yp)}
    print(f'{results_e2e[name]["acc"]:.1%}')

---
## 4. Ergebnisübersicht — 10 s vs. 25 s Fenster

In [ ]:
print(f'10-s-Fenster ({len(X_all)} Samples, Step={STEP_SECS:.0f}s)  vs.  25-s-Fenster (NB05, 256 Samples)')
print()
print(f'{"Modell":6s}  {"S1 (10s)":>9s}  {"S2 (10s)":>9s}  {"E2E (10s)":>10s}  |  {"S2 (25s)":>9s}  {"E2E (25s)":>10s}')
print('─' * 72)

# NB05-Referenzwerte (aus notebook 05 ablesen und hier eintragen)
nb05_s2  = {'RF': 0.900, 'SVM': 0.930, 'kNN': 0.0, 'MLP': 0.0}
nb05_e2e = {'RF': 0.930, 'SVM': 0.0,   'kNN': 0.0, 'MLP': 0.0}

for name in MODELS:
    s1  = results_s1[name]['acc']
    s2  = results_s2[name]['acc']
    e2e = results_e2e[name]['acc']
    r_s2  = f"{nb05_s2[name]:.1%}"  if nb05_s2[name]  > 0 else '  —   '
    r_e2e = f"{nb05_e2e[name]:.1%}" if nb05_e2e[name] > 0 else '  —   '
    print(f'{name:6s}  {s1:>9.1%}  {s2:>9.1%}  {e2e:>10.1%}  |  {r_s2:>9s}  {r_e2e:>10s}')

> **Tipp:** NB05-Referenzwerte oben manuell aus NB05-Output eintragen sobald NB05 gelaufen ist.

---
## 5. Visualisierung

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Links: Balkendiagramm Modellvergleich ─────────────────────────────────────
ax    = axes[0]
names = list(MODELS.keys())
x     = np.arange(len(names))
w     = 0.25
for i, (key, label, res) in enumerate([
    ('S1', 'Stufe 1', results_s1),
    ('S2', 'Stufe 2', results_s2),
    ('E2E', 'End-to-End', results_e2e),
]):
    vals = [res[n]['acc'] for n in names]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=label, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.0%}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel('LOO-Accuracy'); ax.set_ylim(0.6, 1.05)
ax.set_title(f'10-s-Fenster (Step={STEP_SECS:.0f}s)', fontweight='bold')
ax.legend(fontsize=9)

# ── Mitte: Confusion Matrix bestes E2E-Modell ─────────────────────────────────
best = max(results_e2e, key=lambda n: results_e2e[n]['acc'])
r    = results_e2e[best]
cm   = confusion_matrix(r['yt'], r['yp'], labels=CLASSES_FULL)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES_FULL, yticklabels=CLASSES_FULL, ax=axes[1],
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 10, 'weight': 'bold'}, vmin=0)
axes[1].set_title(f'E2E Bestes: {best}  ({r["acc"]:.1%})', fontweight='bold')
axes[1].set_xlabel('Vorhergesagt'); axes[1].set_ylabel('Tatsächlich')

# ── Rechts: Stufe-2-Detail bestes Modell ─────────────────────────────────────
best2 = max(results_s2, key=lambda n: results_s2[n]['acc'])
r2    = results_s2[best2]
cm2   = confusion_matrix(r2['yt'], r2['yp'], labels=CLASSES_FINE)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES_FINE, yticklabels=CLASSES_FINE, ax=axes[2],
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'}, vmin=0)
axes[2].set_title(f'Stufe 2 Bestes: {best2}  ({r2["acc"]:.1%})', fontweight='bold')
axes[2].set_xlabel('Vorhergesagt'); axes[2].set_ylabel('Tatsächlich')

plt.suptitle(f'NB08 — 10-s-Fenster  |  k={K_ME} ME  |  Step={STEP_SECS:.0f}s  |  LOO-CV',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/nb08_10s_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for name in MODELS:
    r = results_e2e[name]
    print(f'{'='*52}')
    print(f'  {name}  —  End-to-End  {r["acc"]:.1%}')
    print(f'{'='*52}')
    print(classification_report(r['yt'], r['yp'], labels=CLASSES_FULL, zero_division=0))